In [2]:
import subprocess
import os
import pandas as pd
import numpy as np
from core.reader import read_ansys_csv, load_experimental_data

ANSYS_EXE_PATH = r"D:\Program Files\ANSYS Inc\ANSYS Student\v252\ansys\bin\winx64\MAPDL.exe" 
WORKING_DIR = os.getcwd()

def run_ansys_simulation(params):
    with open('chab_params.txt', 'w') as f:
        for p in params:
            f.write(f"{p}\n")

    input_file = "chab.mac"
    output_file = "ansys.out"
    
    cmd = [
        ANSYS_EXE_PATH, 
        "-b",
        "-j", "opt_run",
        "-dir", WORKING_DIR, 
        "-i", input_file, 
        "-o", output_file
    ]

    try:
        subprocess.run(cmd, check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print("Ошибка ANSYS:", e)
        return None

    try:
        df_res = read_ansys_csv("chab.csv")
        return df_res
    except Exception as e:
        print(f"Ошибка чтения CSV: {e}")
        return None

In [3]:
real_experiment_data_folder = r'D:\Users\complex_deformations\P29\experimental_vint'
df_exp = load_experimental_data(real_experiment_data_folder)

zero_row = pd.DataFrame(0.0, columns=df_exp.columns, index=[0])
zero_row['Time'] = 1.0
df_exp = pd.concat([zero_row, df_exp]).reset_index(drop=True)

def objective_function(params):
    """
    Считает ошибку между экспериментом и моделью Шабоша.
    params: [sig_y, c1, g1, c2, g2, c3, g3]
    """
    print(f"Simulating: {params}")
    
    df_ansys = run_ansys_simulation(params)
    
    if df_ansys is None or len(df_ansys) != len(df_exp):
        return 1e9
    
    # Считаем MSE по компонентам напряжений
    mse_zz = np.mean((df_ansys['S_ZZ'] - df_exp['S_ZZ'])**2)
    mse_tt = np.mean((df_ansys['S_TT'] - df_exp['S_TT'])**2)
    mse_tz = np.mean((df_ansys['S_TZ'] - df_exp['S_TZ'])**2)
    
    total_error = mse_zz + mse_tt + mse_tz
    print(f"Error: {total_error:.2f}")
    return total_error

In [4]:
import psutil

def kill_ansys_processes():
    for proc in psutil.process_iter():
        if proc.name() in ['ANSYS.exe', 'MAPDL.exe', 'ansys.exe']:
            proc.kill()

kill_ansys_processes()

In [ ]:
# from scipy.optimize import differential_evolution

# # Границы поиска для каждого параметра
# # [sig_y, c1, g1, c2, g2, c3, g3]
# bounds = [
#     (200, 400),      # Sig_Y
#     (1e4, 5e5),      # C1 (Жесткая кинематика)
#     (100, 5000),     # gamma1 (Быстрое насыщение)
#     (1e3, 5e4),      # C2
#     (10, 500),       # gamma2
#     (100, 1e4),      # C3
#     (0, 100)         # gamma3 (Линейная часть)
# ]

# result = differential_evolution(
#     objective_function, 
#     bounds, 
#     strategy='best1bin', 
#     maxiter=50,      # Количество поколений (увеличьте до 20-50 для точности)
#     popsize=15,       # Размер популяции (увеличьте до 10-15)
#     disp=True,
#     workers=1
# )

# print("Оптимальные параметры найдены:")
# print(result.x)

Simulating: [3.75787023e+02 5.67515356e+04 3.13682179e+03 2.78195249e+04
 3.93233530e+02 2.47605727e+02 4.31862636e+01]
Error: 7488.63
Simulating: [3.87873718e+02 1.34046885e+04 1.82106960e+03 4.44449943e+04
 2.06101003e+02 2.48731808e+03 3.90579351e+01]
Error: 19646.18
Simulating: [2.42920996e+02 3.69933368e+05 2.96174660e+03 1.80463957e+04
 3.10816230e+02 9.96564050e+03 3.96199413e+00]
Error: 13774.81
Simulating: [3.25681655e+02 4.38317744e+05 2.56794878e+03 2.91789889e+04
 4.21333397e+02 1.02309403e+03 6.04028544e+01]
Error: 14756.65
Simulating: [2.97944616e+02 4.11172003e+05 5.83659910e+02 4.66022327e+04
 1.75611630e+02 1.42654868e+03 2.50449207e+01]
Error: 758510.58
Simulating: [2.11114518e+02 3.38462408e+05 3.50655062e+03 3.36169610e+04
 3.47793883e+02 2.78356852e+03 5.96075871e+01]
Error: 10837.22
Simulating: [2.45017406e+02 1.07772585e+05 2.40935104e+03 2.59549435e+04
 2.38945549e+02 3.74721818e+02 1.34055627e+01]
Error: 19630.71
Simulating: [3.12264342e+02 2.54806393e+05 1.150

In [ ]:
# from scipy.optimize import minimize

# # Start Point = [sig_y, c1, g1, c2, g2, c3, g3]
# x0 = [
#     216.49263744583521,    # Sig_Y
#     120265.7074049233, # C1
#     623.4456252215341,   # gamma1
#     7828.078591604883,  # C2
#     46.929596510667835,     # gamma2
#     1073.0776477224936,   # C3
#     50.006332654928684      # gamma3
# ]
# print(f"Запуск локальной оптимизации (Nelder-Mead) с начальной точки:\n{x0}")

# bounds = [
#     (200, 400),           # Sig_Y
#     (1e4, 5e5),           # C1
#     (100, 5000),          # gamma1
#     (1e3, 5e4),           # C2
#     (10, 500),            # gamma2
#     (100, 1e4),           # C3
#     (0, 100)             # gamma3
# ]

# res = minimize(
#     objective_function, 
#     x0, 
#     method='Nelder-Mead',
#     bounds=bounds,
#     options={
#         'maxiter': 200,    # Количество запусков ANSYS
#         'disp': True,
#         'xatol': 1.0,     # Точность поиска
#         'fatol': 100.0    # Точность по функции ошибки
#     }
# )

# print("\nОптимизация завершена!")
# print("Лучшие параметры:", res.x)
# print("Лучшая ошибка:", res.fun)

Запуск локальной оптимизации (Nelder-Mead) с начальной точки:
[216.49263744583521, 120265.7074049233, 623.4456252215341, 7828.078591604883, 46.929596510667835, 1073.0776477224936, 50.006332654928684]
Simulating: [2.16492637e+02 1.20265707e+05 6.23445625e+02 7.82807859e+03
 4.69295965e+01 1.07307765e+03 5.00063327e+01]
Error: 1407.01
Simulating: [2.27317269e+02 1.20265707e+05 6.23445625e+02 7.82807859e+03
 4.69295965e+01 1.07307765e+03 5.00063327e+01]
Error: 1661.32
Simulating: [2.16492637e+02 1.26278993e+05 6.23445625e+02 7.82807859e+03
 4.69295965e+01 1.07307765e+03 5.00063327e+01]
Error: 1597.75
Simulating: [2.16492637e+02 1.20265707e+05 6.54617906e+02 7.82807859e+03
 4.69295965e+01 1.07307765e+03 5.00063327e+01]
Error: 1482.74
Simulating: [2.16492637e+02 1.20265707e+05 6.23445625e+02 8.21948252e+03
 4.69295965e+01 1.07307765e+03 5.00063327e+01]
Error: 1441.13
Simulating: [2.16492637e+02 1.20265707e+05 6.23445625e+02 7.82807859e+03
 4.92760763e+01 1.07307765e+03 5.00063327e+01]
Error

C:\Users\Arina\AppData\Local\Temp\ipykernel_6504\2802732262.py:25: RuntimeWarning: Maximum number of iterations has been exceeded.
  res = minimize(


In [5]:
x0 = [
    216.49263744583521,    # Sig_Y
    120265.7074049233, # C1
    623.4456252215341,   # gamma1
    7828.078591604883,  # C2
    46.929596510667835,     # gamma2
    1073.0776477224936,   # C3
    50.006332654928684      # gamma3
]

initial_error = objective_function(x0)
print(f"Ошибка на старте: {initial_error}")

Simulating: [216.49263744583521, 120265.7074049233, 623.4456252215341, 7828.078591604883, 46.929596510667835, 1073.0776477224936, 50.006332654928684]
Ошибка ANSYS: Command '['D:\\Program Files\\ANSYS Inc\\ANSYS Student\\v252\\ansys\\bin\\winx64\\MAPDL.exe', '-b', '-j', 'opt_run', '-dir', 'd:\\Users\\complex_deformations\\P29\\experimental_vint', '-i', 'chab.mac', '-o', 'ansys.out']' returned non-zero exit status 100.
Ошибка на старте: 1000000000.0
